In [ ]:
import pandas as pd

roll_number = "1024170007"

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

categories = ["billing", "account", "general"]

last_two_digits = [int(roll_number[-2]), int(roll_number[-1])]

personalized_entries = []

for d in last_two_digits:
    category = categories[d % 3]

    if category == "billing":
        entry = {
            "question": "how can i check my pending payment",
            "answer": "You can check pending payments in the billing section.",
            "keywords": "pending payment bill",
            "category": "billing"
        }

    elif category == "account":
        entry = {
            "question": "how do i update my registered mobile number",
            "answer": "Go to account settings and update your registered mobile number.",
            "keywords": "mobile update account",
            "category": "account"
        }

    else:
        entry = {
            "question": "where can i get customer support",
            "answer": "You can contact customer support through the help section.",
            "keywords": "support help contact",
            "category": "general"
        }

    personalized_entries.append(entry)

df = pd.DataFrame(fixed_entries + personalized_entries)

print(df)

In [ ]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        question_words = set(row["question"].lower().split())
        keyword_words = set(row["keywords"].lower().split())

        question_matches = len(query_words.intersection(question_words))
        keyword_matches = len(query_words.intersection(keyword_words))

        score = question_matches + keyword_matches

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    result_df = pd.DataFrame(results)

    if not result_df.empty:
        result_df = result_df.sort_values(
            by="score",
            ascending=False
        )

    return result_df


query = "how can i pay fee"

result = score_query(query, df)

print("Query:", query)
print(result)

In [ ]:
def same_category(category_name, df):
    result = df[df["category"] == category_name]

    return result["question"]


personalized_category = df.iloc[4]["category"]

print("Category:", personalized_category)
print("\nQuestions:")
print(same_category(personalized_category, df))

In [ ]:
selected_index = 0

new_keyword = "subscription"

df.loc[selected_index, "keywords"] = (
    df.loc[selected_index, "keywords"] + " " + new_keyword
)

filename = "1024170007_faq_data.csv"

df.to_csv(filename, index=False)

print("Updated entry:")
print(df.loc[selected_index])

print("\nFile saved as:", filename)

In [ ]:
category_counts = df.groupby("category").size()

print("FAQ entries per category:")
print(category_counts)

In [ ]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():
        question_words = set(row["question"].lower().split())
        keyword_words = set(row["keywords"].lower().split())

        question_matches = len(
            query_words.intersection(question_words)
        )

        keyword_matches = len(
            query_words.intersection(keyword_words)
        )

        score = question_matches + keyword_matches

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if len(results) == 0:
        print("No matching FAQ found.")
        return

    result_df = pd.DataFrame(results)

    highest_score = result_df["score"].max()

    best_matches = result_df[
        result_df["score"] == highest_score
    ]

    if len(best_matches) > 1:
        print("Multiple equally good matches found:")
    else:
        print("Best match found:")

    print(best_matches)


tie_query = "fee"

print("TIE QUERY:")
print("Query:", tie_query)
score_query_with_ties(tie_query, df)


print("\n-----------------------------\n")


non_tie_query = "password reset login"

print("NON-TIE QUERY:")
print("Query:", non_tie_query)
score_query_with_ties(non_tie_query, df)